# NASA Near-Earth Object Hazard Classifier

This project uses NASA’s Near-Earth Object (NeoWs) API to retrieve asteroid data and train a Random Forest classifier to predict whether an asteroid is potentially hazardous.


In [ ]:
import os
import json
import requests
import pandas as pd
import matplotlib.pyplot as plt


## Project Overview

This project uses NASA’s NeoWs API to retrieve near-Earth object data and train a classification model to predict whether an asteroid may be potentially hazardous. The workflow retrieves the data, converts the JSON response into a Pandas DataFrame, preprocesses the data, and trains a Random Forest classifier to predict `is_potentially_hazardous_asteroid`.

The model is intended as a way to identify asteroids that may warrant closer inspection. Its predictions reflect patterns in the supplied NASA data and should not be treated as an independent determination of actual impact risk.


## Step 1: Retrieve NASA Data

The NASA API has rate limits, so retrieving the full dataset can take several minutes.


In [ ]:
# This is the API key I was given. I wonder why they are so long. Even for encryption standards, this is crazy:
import os
# Set the NASA_API_KEY environment variable, or paste your own key in place of the placeholder
NASA_API_KEY = os.environ.get("NASA_API_KEY", "YOUR_NASA_API_KEY_HERE")

In [ ]:
# configure the URL and all the parameters. Page: 0 pretty much means "start at page 0"; size means 
# "return 20 asteroids to each request, which is the limit.
asteroids = []
url = "https://api.nasa.gov/neo/rest/v1/neo/browse"
params = {"api_key": NASA_API_KEY, "page": 0, "size": 20}

# Create where the data will be stored to, and save the total number of pages for the request loop
NASA_response = requests.get(url, params=params).json()
asteroids.extend(NASA_response["near_earth_objects"])
total_pages = NASA_response["page"]["total_pages"]

# Loop over the last 500 pages (~10000 asteroids). I would love to do more, but there are rate limits,
# and there are also limits to the total amount of data you can request per day. Therefore, I didn't 
# want to request too much, find a bug, and then have to create a new API key.
for page in range(max(0, total_pages - 500), total_pages):
    params["page"] = page
    response = requests.get(url, params=params).json()
    asteroids.extend(response["near_earth_objects"])
    print(f"Retrieved Page {page + 1} of {total_pages}", end="\r")

print(f"\nTotal asteroids: {len(asteroids)}")

Now that I have a list with all the asteroids, I want to take a look at some of the data. 

Note: I didn't include all tests, but I did include all the ones I got the important information from.

In [ ]:
# Look at the keys. 
print(asteroids[0].keys())

In [ ]:
# Look at the structure (this took a while to sift through, I probably could have done it more efficiently, 
# but it worked out in the end)
print(asteroids[0])

## Step 2: Convert the API Data to a DataFrame



In [ ]:
rows = []
for a in asteroids:
    # This was the only weird one, I just took the closest it has ever come
    cad = min(a["close_approach_data"], key=lambda x: float(x["miss_distance"]["kilometers"]), default={})

    # The only data that could potentially have None is accounted for. However, when manually importing like this, it is
    # important to be vigilant about. I got a couple of errors the first time around because there were a few missing data points
    # I didn't account for.

    # Also note that there are a few pieces of data that might matter, but I viewed these as the most significant. The accuracy turned out to be 
    # really good anyway!
    rows.append({
        "id": a["id"],
        "name": a["name"],
        "is_hazardous": int(a["is_potentially_hazardous_asteroid"]),
        "absolute_magnitude": a.get("absolute_magnitude_h", None),
        "est_diameter_min_km": a.get("estimated_diameter", {}).get("kilometers", {}).get("estimated_diameter_min", None),
        "est_diameter_max_km": a.get("estimated_diameter", {}).get("kilometers", {}).get("estimated_diameter_max", None),
        "relative_velocity_km_s": float(cad.get("relative_velocity", {}).get("kilometers_per_second", 0)),
        "miss_distance_km": float(cad.get("miss_distance", {}).get("kilometers", 0))
    })

# Get rid of any duplicate asteroids and display the first few just to check things. It should look nice.
df = pd.DataFrame(rows).drop_duplicates().reset_index()
df.head()

In good practice, I want to check for null data:

In [ ]:
# See if there is any null data
df.isnull().sum()

Alright, there are a few with null data. However, as it is so few, I will just remove them:

In [ ]:
# Drop it and check again
df = df.dropna()
df.isnull().sum()

## Step 3: Train the Classifier



In [ ]:
# Separating training vs test data is important, which logically makes it the first step

from sklearn.model_selection import train_test_split

# name, id, and is_hazardous should not be included for training, so I selected only the valid training features
features = [
    "absolute_magnitude",
    "est_diameter_min_km",
    "est_diameter_max_km",
    "relative_velocity_km_s",
    "miss_distance_km"
]

x = df[features]
y = df["is_hazardous"]

# Split!
x_train,x_validate,y_train,y_validate=train_test_split(x,y,test_size=0.2,random_state=123456)

Now, I just wanted to see what everything looks like.

In [ ]:
x_train.info()

In [ ]:
x_validate.info()

In [ ]:
y_train.value_counts()

In [ ]:
y_validate.value_counts()

The classifier uses class weighting because correctly identifying potentially hazardous asteroids is especially important. The hazardous class is weighted more heavily through the `class_weight` parameter.

Sources:

https://www.geeksforgeeks.org/machine-learning/how-does-the-classweight-parameter-in-scikit-learn-work/

https://scikit-learn.org/stable/modules/generated/sklearn.utils.class_weight.compute_class_weight.html


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Initialize the classifier. I used largely the same parameters from the last assignment.
AsteroidRF = RandomForestClassifier(
    n_estimators=150,       
    max_depth=16,           
    min_samples_split=25,    
    min_samples_leaf=8,
    random_state=314159,   
    class_weight = {0: 1, 1: 50}, # See markup above
    n_jobs=-1 
)

AsteroidRF.fit(x_train, y_train)

In [ ]:
y_predict = AsteroidRF.predict(x_validate)

## Step 4: Evaluate the Model



In [ ]:
# Test accuracy 
# Note: This accuracy would be FAR higher if I didn't weight the hazardous asteroids more (around 97-99%).
# However, the accuracy for hazardous asteroid identification is much better.

print(f'Overall accuracy against training data is {AsteroidRF.score(x_train, y_train):5.2%}')
print(f'Overall accuracy against validation data is {AsteroidRF.score(x_validate, y_validate):5.2%}')

from sklearn.metrics import classification_report
print(classification_report(y_validate, y_predict))

In [ ]:
# This communicates a more complete story of why this may be used. While there are some incorrect predictions, 
# by and large, the weighting on the hazardous asteroids had a really profound effect.

pd.set_option("display.max_rows", 75)
comparison = pd.DataFrame({"actual": y_validate, "predicted": y_predict})
comparison[comparison["actual"] == 1]

In [ ]:
# Have you ever seen a confusion matrix? I assume so, but when I was looking for good ways to visualize data,
# this really stuck out to me. Pretty much (ONLY FOR PERCENTS), the diagonals should be as dark as possible, and the off-diagonals
# should be as white as possible. The fact that the bottom left square is darker than the top right is an effect
# created by the weighting done earlier; there are far fewer incorrect hazardous predictions.

# For the numeric one, this is just a way of displaying the numbers in all the categories. I will also add that I already knew
# about this type of graph from when I did the M3 Junior Challenge. 

# Sources:

# https://www.geeksforgeeks.org/machine-learning/how-to-plot-confusion-matrix-with-labels-in-sklearn/
# https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html
# https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Create two confusion matrices, one for the percentages, one to show the samples/individuals

cm = confusion_matrix(y_validate, y_predict, normalize="true")
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Not Hazardous", "Hazardous"])
disp.plot(cmap="Blues", values_format=".1%")
disp.ax_.set_xlabel("Predicted")
disp.ax_.set_ylabel("Actual")
plt.title("Confusion Matrix Percents")
plt.show()

cm = confusion_matrix(y_validate, y_predict)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Not Hazardous", "Hazardous"])
disp.plot(cmap="Greens")
disp.ax_.set_xlabel("Predicted")
disp.ax_.set_ylabel("Actual")
plt.title("Confusion Matrix Individuals")
plt.show()